# Unsloth ORPO Full Training (Colab)

This notebook is the **Act III full training run** for Path B. It trains on the prepared public-only preference file at `training_data/orpo_preferences.jsonl`, saves a LoRA adapter, and writes the run artifacts needed for Act IV.

## Run Notes

- Use a Colab GPU runtime.
- Run the install cell first, then **restart the Colab session** before continuing.
- This notebook is for the real ORPO corpus, not the 5-row smoke file.
- Default backbone is `unsloth/Qwen3-0.6B` in 16-bit LoRA mode, which matches the repo PRD.
- The install step below force-reinstalls both `unsloth` and `unsloth_zoo` together to avoid the import mismatch you hit in Colab.


In [ ]:
# Install Unsloth and training dependencies in a version-synced way.
# After this cell finishes, use Runtime -> Restart session, then continue from the next cell.

!pip uninstall -y unsloth unsloth_zoo
!pip install --upgrade pip
!pip install --upgrade --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install --upgrade --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth-zoo.git
!pip install --upgrade xformers trl peft accelerate bitsandbytes datasets matplotlib


In [ ]:
import importlib.metadata as metadata
import platform
import sys
import torch

packages = ["unsloth", "unsloth_zoo", "trl", "transformers", "torch", "datasets", "peft", "accelerate", "bitsandbytes"]
versions = {}
for package in packages:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = "not-installed"

print({
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "packages": versions,
})

!nvidia-smi

print("cuda available:", torch.cuda.is_available())
print("cuda device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("gpu: none")

assert torch.cuda.is_available(), (
    "CUDA is not available. In Colab, go to Runtime -> Change runtime type -> T4 GPU, "
    "save, restart the session, and rerun from the top."
)


## Repo Access

Pick one mode:

1. `drive`: use a copy already stored in Google Drive.
2. `clone`: clone the repo directly in Colab.

Set the values below before running the cell.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_MODE = "clone"  # "drive" or "clone"
DRIVE_REPO_PATH = "/content/drive/MyDrive/Sales Eval Bench"
CLONE_URL = "https://github.com/nebiyou27/sales-eval-bench.git"
CLONE_DIR = "/content/Sales Eval Bench"

if REPO_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    repo_root = Path(DRIVE_REPO_PATH)
elif REPO_MODE == "clone":
    repo_root = Path(CLONE_DIR)
    if not repo_root.exists():
        subprocess.run(["git", "clone", CLONE_URL, str(repo_root)], check=True)
else:
    raise ValueError(f"Unsupported REPO_MODE: {REPO_MODE}")

assert repo_root.exists(), f"Repo root not found: {repo_root}"
os.chdir(repo_root)
print(f"repo_root={repo_root}")

## Load And Validate ORPO Training Data

This is the public-only full preference file created during Act III prep. The checks below make sure we are not accidentally training on smoke or held-out data.

In [ ]:
import json
from datasets import Dataset

FULL_DATASET_PATH = repo_root / "training_data" / "orpo_preferences.jsonl"
SMOKE_DATASET_PATH = repo_root / "tenacious_bench_v0.1" / "smoke" / "dummy_orpo_preferences.jsonl"

assert FULL_DATASET_PATH.exists(), f"Full ORPO dataset not found: {FULL_DATASET_PATH}"
assert SMOKE_DATASET_PATH.exists(), f"Smoke dataset missing unexpectedly: {SMOKE_DATASET_PATH}"
assert FULL_DATASET_PATH != SMOKE_DATASET_PATH, "This notebook must use the full ORPO dataset, not the smoke file."

records = [json.loads(line) for line in FULL_DATASET_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
print({"dataset_path": str(FULL_DATASET_PATH), "records_loaded": len(records)})

required_keys = {"prompt", "chosen", "rejected", "source_partition", "output_partition"}
for index, record in enumerate(records, start=1):
    missing = required_keys - set(record)
    assert not missing, f"Record {index} is missing keys: {missing}"
    assert record["chosen"] != record["rejected"], f"Record {index} has identical chosen and rejected strings"
    assert record["source_partition"] in {"train", "dev"}, f"Unexpected source_partition in record {index}: {record['source_partition']}"
    assert record["output_partition"] != "held_out", f"Held-out leakage found in record {index}"

full_ds = Dataset.from_list([
    {
        "prompt": record["prompt"],
        "chosen": record["chosen"],
        "rejected": record["rejected"],
    }
    for record in records
])

source_partition_counts = {}
for record in records:
    key = record["source_partition"]
    source_partition_counts[key] = source_partition_counts.get(key, 0) + 1

print({
    "dataset_rows": len(full_ds),
    "source_partition_counts": source_partition_counts,
})
print(full_ds[0])

## Model And LoRA Setup

The defaults here follow the repo PRD for the real Path B run:

- `unsloth/Qwen3-0.6B`
- 16-bit LoRA, not QLoRA
- `r=16`, `alpha=16`, `dropout=0`
- `max_seq_length=1024`

Only run the import cell below **after** the install cell has finished and the Colab session has been restarted.


In [ ]:
import torch
from unsloth import FastLanguageModel

cuda_available = torch.cuda.is_available()
assert cuda_available, "CUDA is not available. Switch Colab to a GPU runtime and rerun from the top."

MODEL_NAME = "unsloth/Qwen3-0.6B"
MAX_SEQ_LENGTH = 1024
LOAD_IN_4BIT = False

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)

print({
    "model_name": MODEL_NAME,
    "max_seq_length": MAX_SEQ_LENGTH,
    "load_in_4bit": LOAD_IN_4BIT,
    "device": torch.cuda.get_device_name(0),
})


In [ ]:
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0
TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print({
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "target_modules": TARGET_MODULES,
})

## ORPO Full Training Run

The defaults below are meant for the first real T4 run: one epoch over the 211-row public preference file, batch size 1 with accumulation, and regular checkpointing. After this first run, you can tune steps or learning rate based on the loss curve.

In [ ]:
import inspect
from trl import ORPOConfig, ORPOTrainer

OUTPUT_DIR = repo_root / "models" / "orpo_qwen3_0_6b_full_run"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

orpo_args = ORPOConfig(
    output_dir=str(OUTPUT_DIR),
    learning_rate=2e-5,
    beta=0.1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    warmup_steps=5,
    fp16=True,
    bf16=False,
    logging_steps=5,
    save_strategy="steps",
    save_steps=25,
    optim="adamw_8bit",
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=512,
    remove_unused_columns=False,
    report_to=[],
)

trainer_kwargs = {
    "model": model,
    "args": orpo_args,
    "train_dataset": full_ds,
}

trainer_signature = inspect.signature(ORPOTrainer.__init__)
if "tokenizer" in trainer_signature.parameters:
    trainer_kwargs["tokenizer"] = tokenizer
if "processing_class" in trainer_signature.parameters:
    trainer_kwargs["processing_class"] = tokenizer

trainer = ORPOTrainer(**trainer_kwargs)
train_result = trainer.train()

loss_values = [
    entry["loss"]
    for entry in trainer.state.log_history
    if isinstance(entry, dict) and "loss" in entry
]

print({
    "global_step": trainer.state.global_step,
    "training_loss": getattr(train_result, "training_loss", None),
    "logged_loss_count": len(loss_values),
    "max_memory_allocated_gb": round(torch.cuda.max_memory_allocated() / (1024 ** 3), 2),
    "output_dir": str(OUTPUT_DIR),
})

In [ ]:
import datetime as dt
import math
import matplotlib.pyplot as plt

RUN_ARTIFACT_DIR = repo_root / "models" / "orpo_qwen3_0_6b_full_adapter"
RUN_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(str(RUN_ARTIFACT_DIR))
tokenizer.save_pretrained(str(RUN_ARTIFACT_DIR))

loss_points = [entry for entry in trainer.state.log_history if isinstance(entry, dict) and "loss" in entry]
loss_steps = [entry.get("step") for entry in loss_points]
loss_values = [float(entry["loss"]) for entry in loss_points]

assert loss_values, "No loss values were logged during training."
assert all(math.isfinite(loss) for loss in loss_values), f"Non-finite loss detected: {loss_values}"

plt.figure(figsize=(10, 5))
plt.plot(loss_steps, loss_values, marker="o")
plt.title("ORPO Training Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.grid(True, linestyle="--", alpha=0.5)
loss_curve_path = RUN_ARTIFACT_DIR / "loss_curve.png"
plt.savefig(loss_curve_path, bbox_inches="tight")
plt.show()

training_run_summary = {
    "timestamp_utc": dt.datetime.now(dt.UTC).replace(microsecond=0).isoformat().replace("+00:00", "Z"),
    "status": "pass",
    "dataset_path": str(FULL_DATASET_PATH),
    "record_count": len(records),
    "model_name": MODEL_NAME,
    "precision": "fp16",
    "qlora": LOAD_IN_4BIT,
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "orpo_beta": 0.1,
    "learning_rate": 2e-5,
    "batch_size": 1,
    "gradient_accumulation_steps": 4,
    "num_train_epochs": 1,
    "global_step": trainer.state.global_step,
    "loss_values": loss_values,
    "all_losses_finite": True,
    "adapter_dir": str(RUN_ARTIFACT_DIR),
    "loss_curve_path": str(loss_curve_path),
    "max_memory_allocated_gb": round(torch.cuda.max_memory_allocated() / (1024 ** 3), 2),
}

summary_path = RUN_ARTIFACT_DIR / "training_run_summary.json"
summary_path.write_text(json.dumps(training_run_summary, indent=2), encoding="utf-8")

print("RUN_ARTIFACT_DIR:", RUN_ARTIFACT_DIR)
print("Files:", sorted(p.name for p in RUN_ARTIFACT_DIR.iterdir()))
print(json.dumps(training_run_summary, indent=2))

## After The Run

Keep these artifacts for Act IV:

- `adapter_config.json`
- `adapter_model.safetensors`
- tokenizer files
- `training_run_summary.json`
- `loss_curve.png`

Then record the run in `docs/progress.md` with the date, model, dataset row count, runtime result, peak VRAM, and any notes before moving on to held-out ablations.

If the `from unsloth import FastLanguageModel` import ever fails again with a mismatch error, rerun the install cell, restart the Colab session, and continue from the GPU/version check cell.
